# 🧠 Diagnóstico Completo — CAP Sleep Database

Este notebook realiza un análisis exploratorio completo de la base de datos antes de cualquier preprocesamiento.

**Contenido:**
1. Configuración y mapeo de canales
2. Inventario de canales por paciente (matriz presencia/ausencia)
3. Disponibilidad por categoría de señal (EEG, EMG, ECG…)
4. Análisis de saturación por canal
5. Frecuencias de muestreo y duración de grabaciones
6. Recomendaciones de preprocesamiento
7. Exportar resultados a CSV

## 0 · Imports y configuración

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from pathlib import Path

mne.set_log_level('ERROR')
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# ── AJUSTA ESTA RUTA ──────────────────────────────────────
BASE_PATH = Path.cwd() / '../data/cap-sleep-database'
# ─────────────────────────────────────────────────────────

SATURATION_THRESHOLD = 950   # µV
OUT_DIR = BASE_PATH.parent / 'diagnostico_outputs'
OUT_DIR.mkdir(exist_ok=True)

edf_files = sorted(BASE_PATH.glob('*.edf'))
print(f'Archivos .edf encontrados: {len(edf_files)}')
for f in edf_files[:5]:
    print(f'  {f.name}')
if len(edf_files) > 5:
    print(f'  ... y {len(edf_files)-5} más')

## 1 · Mapeo de canales → categoría

Si al correr el notebook aparecen canales en la sección **"CANALES SIN MAPEAR"**, agrégalos aquí y vuelve a ejecutar.

In [ ]:
CHANNEL_MAPPING = {
    # ── EEG ────────────────────────────────────────────────
    'FP1-F3': 'EEG', 'FP2-F4': 'EEG', 'F3-C3': 'EEG',
    'F4-C4': 'EEG', 'C3-P3': 'EEG', 'C4-P4': 'EEG',
    'P3-O1': 'EEG', 'P4-O2': 'EEG', 'F7-T3': 'EEG',
    'F8-T4': 'EEG', 'T3-T5': 'EEG', 'T4-T6': 'EEG',
    'FP1-A1': 'EEG', 'FP2-A2': 'EEG', 'F3-A1': 'EEG',
    'F4-A2': 'EEG', 'C3-A1': 'EEG', 'C4-A2': 'EEG',
    'C3-A2': 'EEG', 'C4-A1': 'EEG', 'P3-A1': 'EEG',
    'P4-A2': 'EEG', 'O1-A1': 'EEG', 'O2-A2': 'EEG',
    'FZ-CZ': 'EEG', 'CZ-PZ': 'EEG',
    'F3-CZ': 'EEG', 'F4-CZ': 'EEG',

    # ── EMG ────────────────────────────────────────────────
    'EMG1-EMG2': 'EMG', 'EMG': 'EMG',
    'CHIN1-CHIN2': 'EMG', 'CHIN': 'EMG',
    'LAT1-LAT2': 'EMG', 'RAT1-RAT2': 'EMG',
    'LAT': 'EMG', 'RAT': 'EMG',

    # ── EOG ────────────────────────────────────────────────
    'ROC-LOC': 'EOG', 'LOC-A2': 'EOG', 'ROC-A1': 'EOG',
    'EOG': 'EOG', 'E1-M2': 'EOG', 'E2-M2': 'EOG',

    # ── ECG ────────────────────────────────────────────────
    'ECG1-ECG2': 'ECG', 'ECG': 'ECG', 'EKG': 'ECG',
    'ECG2-ECG3': 'ECG',

    # ── Flujo nasal ─────────────────────────────────────────
    'FLUSSO': 'FLOW', 'FLOW': 'FLOW',
    'CANNULA': 'FLOW', 'NASAL': 'FLOW', 'AIRFLOW': 'FLOW',

    # ── Tórax ───────────────────────────────────────────────
    'TORACE': 'THORAX', 'TORACICO': 'THORAX',
    'THORAX': 'THORAX', 'CHEST': 'THORAX',

    # ── Abdomen ─────────────────────────────────────────────
    'ADDOME': 'ABDOMEN', 'ADDDOME': 'ABDOMEN',
    'ABDO': 'ABDOMEN', 'ABDOMEN': 'ABDOMEN',

    # ── Oximetría ──────────────────────────────────────────
    'SAO2': 'OXYGEN', 'SPO2': 'OXYGEN',
    'PLETH': 'OXYGEN', 'OX STATUS': 'OXYGEN',

    # ── Otros ──────────────────────────────────────────────
    'DX1-DX2': 'OTHER', 'SX1-SX2': 'OTHER',
    'POSITION': 'OTHER', 'SNORE': 'OTHER',
    'MICROPHONE': 'OTHER', 'MIC': 'OTHER',
}

CATEGORY_COLORS = {
    'EEG':     '#4C72B0',
    'EMG':     '#DD8452',
    'EOG':     '#55A868',
    'ECG':     '#C44E52',
    'FLOW':    '#8172B3',
    'THORAX':  '#937860',
    'ABDOMEN': '#DA8BC3',
    'OXYGEN':  '#8C8C8C',
    'OTHER':   '#CCB974',
    'UNKNOWN': '#BBBBBB',
}

def normalize(name):
    return name.strip().upper()

def categorize(name):
    return CHANNEL_MAPPING.get(normalize(name), 'UNKNOWN')

print(f'Categorías definidas: {sorted(set(CHANNEL_MAPPING.values()))}')
print(f'Canales mapeados: {len(CHANNEL_MAPPING)}')

## 2 · Inventario de canales por paciente

In [ ]:
records = []
total = len(edf_files)

for i, file in enumerate(edf_files, 1):
    print(f'  [{i:>3}/{total}] {file.stem}', end='\r')
    try:
        raw = mne.io.read_raw_edf(file, preload=False, verbose=False)
        for ch in raw.ch_names:
            records.append({
                'patient':  file.stem,
                'channel':  normalize(ch),
                'category': categorize(ch),
            })
    except Exception as e:
        print(f'\n  ⚠️  Error en {file.stem}: {e}')

channel_df = pd.DataFrame(records)
print(f'\n✅ Inventario completado — {len(channel_df)} entradas totales')
channel_df.head(10)

In [ ]:
# Matriz paciente × canal (0 / 1)
pivot_df = (
    channel_df
    .drop_duplicates(subset=['patient', 'channel'])
    .assign(present=1)
    .pivot(index='patient', columns='channel', values='present')
    .fillna(0).astype(int)
)

print(f'Dimensiones de la matriz: {pivot_df.shape}  (pacientes × canales únicos)')
pivot_df.head()

In [ ]:
# Canales sin mapear
unknown = (
    channel_df[channel_df['category'] == 'UNKNOWN']['channel']
    .value_counts()
)

if unknown.empty:
    print('✅ Todos los canales están mapeados.')
else:
    print(f'⚠️  {len(unknown)} canales sin mapear (agrégalos al CHANNEL_MAPPING):\n')
    print(unknown.to_string())

## 3 · Disponibilidad por categoría de señal

In [ ]:
n_patients = channel_df['patient'].nunique()

cat_summary = (
    channel_df
    .drop_duplicates(subset=['patient', 'category'])
    .groupby('category')['patient'].nunique()
    .reset_index(name='n_patients')
    .assign(pct=lambda x: (x['n_patients'] / n_patients * 100).round(1))
    .sort_values('n_patients', ascending=False)
)

cat_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

colors = [CATEGORY_COLORS.get(c, '#BBBBBB') for c in cat_summary['category']]
bars = ax.barh(cat_summary['category'], cat_summary['pct'], color=colors, edgecolor='white', height=0.6)

for bar, (_, row) in zip(bars, cat_summary.iterrows()):
    ax.text(
        bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
        f"{row['n_patients']} pac. ({row['pct']}%)",
        va='center', ha='left', fontsize=9, color='#333'
    )

ax.axvline(100, color='#999', lw=1, ls='--', alpha=0.6)
ax.set_xlim(0, 125)
ax.set_xlabel('% de pacientes con al menos un canal de esta categoría')
ax.set_title('Disponibilidad de señales por categoría', fontsize=13, fontweight='bold')
ax.invert_yaxis()
sns.despine(left=True)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: paciente × categoría
cat_pivot = (
    channel_df
    .drop_duplicates(subset=['patient', 'category'])
    .assign(present=1)
    .pivot(index='patient', columns='category', values='present')
    .fillna(0).astype(int)
)

fig, ax = plt.subplots(figsize=(10, max(4, len(cat_pivot) * 0.22)))
sns.heatmap(
    cat_pivot, cmap=['#F0F0F0', '#4C72B0'],
    linewidths=0.3, linecolor='white',
    cbar=False, ax=ax
)
ax.set_title('Presencia de categorías por paciente (azul = disponible)', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Paciente')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 4 · Análisis de saturación por canal

Muestras con |valor| ≥ `SATURATION_THRESHOLD` µV se consideran saturadas (artefactos de clipping del amplificador).

In [ ]:
sat_records = []
total = len(edf_files)

for i, file in enumerate(edf_files, 1):
    print(f'  [{i:>3}/{total}] {file.stem}', end='\r')
    try:
        raw = mne.io.read_raw_edf(file, preload=True, verbose=False)
        data_uv = raw.get_data() * 1e6   # Volts → µV

        for j, ch in enumerate(raw.ch_names):
            ch_data = data_uv[j]
            n_total = len(ch_data)
            n_sat   = int(np.sum(np.abs(ch_data) >= SATURATION_THRESHOLD))
            sat_records.append({
                'patient':       file.stem,
                'channel':       normalize(ch),
                'category':      categorize(ch),
                'n_samples':     n_total,
                'n_saturated':   n_sat,
                'pct_saturated': round(n_sat / n_total * 100, 4) if n_total > 0 else 0.0,
            })
    except Exception as e:
        print(f'\n  ⚠️  Error en {file.stem}: {e}')

sat_df = pd.DataFrame(sat_records)
print(f'\n✅ Saturación calculada — {len(sat_df)} entradas')

In [ ]:
# Agregado por canal
sat_agg = (
    sat_df
    .groupby(['channel', 'category'])
    .agg(
        n_patients        = ('patient', 'nunique'),
        mean_pct_sat      = ('pct_saturated', 'mean'),
        max_pct_sat       = ('pct_saturated', 'max'),
        patients_with_sat = ('pct_saturated', lambda x: (x > 0).sum()),
    )
    .round(4)
    .reset_index()
    .sort_values('mean_pct_sat', ascending=False)
)

# Marcar canales problemáticos
sat_agg['flag'] = sat_agg['mean_pct_sat'].apply(lambda x: '⚠️' if x > 1.0 else '')

sat_agg.head(20)

In [ ]:
# Gráfico: top canales con mayor saturación promedio
top_sat = sat_agg[sat_agg['mean_pct_sat'] > 0].head(20)

if top_sat.empty:
    print('✅ No se detectó saturación en ningún canal.')
else:
    fig, ax = plt.subplots(figsize=(9, max(3, len(top_sat) * 0.35)))
    colors = [CATEGORY_COLORS.get(c, '#BBB') for c in top_sat['category']]
    bars = ax.barh(top_sat['channel'], top_sat['mean_pct_sat'], color=colors, edgecolor='white', height=0.65)

    for bar, (_, row) in zip(bars, top_sat.iterrows()):
        ax.text(
            bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
            f"{row['mean_pct_sat']:.2f}%  [{row['category']}]",
            va='center', ha='left', fontsize=8, color='#444'
        )

    ax.axvline(1.0, color='red', lw=1.2, ls='--', alpha=0.7, label='Umbral 1%')
    ax.set_xlabel('Saturación promedio (%)')
    ax.set_title(f'Saturación por canal (umbral ±{SATURATION_THRESHOLD} µV)', fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.legend(fontsize=8)
    sns.despine(left=True)
    ax.tick_params(left=False)
    plt.tight_layout()
    plt.show()

## 5 · Frecuencias de muestreo y duración de grabaciones

In [ ]:
sfreq_records = []
total = len(edf_files)

for i, file in enumerate(edf_files, 1):
    print(f'  [{i:>3}/{total}] {file.stem}', end='\r')
    try:
        raw = mne.io.read_raw_edf(file, preload=False, verbose=False)
        sfreq_records.append({
            'patient':      file.stem,
            'sfreq_hz':     raw.info['sfreq'],
            'n_channels':   len(raw.ch_names),
            'duration_min': round(raw.times[-1] / 60, 2),
            'n_samples':    raw.n_times,
        })
    except Exception as e:
        print(f'\n  ⚠️  Error en {file.stem}: {e}')

sfreq_df = pd.DataFrame(sfreq_records)
print(f'\n✅ Frecuencias calculadas')
sfreq_df.describe()

In [ ]:
freq_counts = sfreq_df['sfreq_hz'].value_counts().sort_index()
print('Distribución de frecuencias de muestreo:')
print(freq_counts.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Frecuencias de muestreo
axes[0].bar(freq_counts.index.astype(str), freq_counts.values, color='#4C72B0', edgecolor='white')
axes[0].set_xlabel('Frecuencia de muestreo (Hz)')
axes[0].set_ylabel('N° de pacientes')
axes[0].set_title('Distribución de frecuencias de muestreo', fontweight='bold')
sns.despine(ax=axes[0])

# Duración
axes[1].hist(sfreq_df['duration_min'], bins=20, color='#55A868', edgecolor='white')
axes[1].set_xlabel('Duración (minutos)')
axes[1].set_ylabel('N° de pacientes')
axes[1].set_title('Distribución de duración de grabaciones', fontweight='bold')
axes[1].axvline(sfreq_df['duration_min'].mean(), color='red', ls='--', lw=1.2, label=f"Media: {sfreq_df['duration_min'].mean():.0f} min")
axes[1].legend(fontsize=9)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.show()

## 6 · Recomendaciones de preprocesamiento

In [ ]:
issues = []
ok     = []

for cat in ['EEG', 'EMG', 'ECG']:
    row = cat_summary[cat_summary['category'] == cat]
    if row.empty:
        issues.append(f'❌  {cat} no encontrado en ningún paciente.')
    elif row.iloc[0]['pct'] < 100:
        issues.append(f'⚠️   {cat} solo disponible en {row.iloc[0]["pct"]}% de los pacientes. '
                      f'Decide si excluir pacientes o imputar.')
    else:
        ok.append(f'✅  {cat} disponible en el 100% de los pacientes.')

high_sat = sat_agg[sat_agg['mean_pct_sat'] > 1.0]
if not high_sat.empty:
    canales = ', '.join(high_sat['channel'].tolist())
    issues.append(f'⚠️   Saturación >1% promedio en: {canales}.\n'
                  f'     → Considera rechazo de épocas o interpolación.')
else:
    ok.append('✅  Saturación dentro de rangos aceptables (<1%) en todos los canales.')

if freq_counts.shape[0] > 1:
    freqs = ', '.join([str(int(f)) for f in freq_counts.index])
    modal = int(freq_counts.idxmax())
    issues.append(f'⚠️   Múltiples frecuencias de muestreo: {freqs} Hz.\n'
                  f'     → Resamplea todo a {modal} Hz antes de extraer características.')
else:
    ok.append(f'✅  Frecuencia de muestreo uniforme: {int(freq_counts.index[0])} Hz.')

unknown_count = (channel_df['category'] == 'UNKNOWN').sum()
if unknown_count > 0:
    issues.append(f'⚠️   {unknown_count} entradas de canal sin categoría.\n'
                  f'     → Revisa la celda 2 y amplía CHANNEL_MAPPING.')
else:
    ok.append('✅  Todos los canales están correctamente mapeados.')

print('=' * 55)
print('  DIAGNÓSTICO FINAL')
print('=' * 55)
for msg in ok:
    print(f'  {msg}')
print()
for msg in issues:
    for line in msg.split('\n'):
        print(f'  {line}')
    print()
print('=' * 55)

## 7 · Exportar resultados a CSV

In [ ]:
pivot_df.to_csv(OUT_DIR / '01_matriz_canales.csv')
cat_summary.to_csv(OUT_DIR / '02_resumen_categorias.csv', index=False)
sat_agg.to_csv(OUT_DIR / '03_saturacion_por_canal.csv', index=False)
sat_df.to_csv(OUT_DIR / '04_saturacion_por_paciente.csv', index=False)
sfreq_df.to_csv(OUT_DIR / '05_frecuencias_muestreo.csv', index=False)
channel_df.to_csv(OUT_DIR / '06_inventario_canales_raw.csv', index=False)

print(f'💾 CSVs guardados en: {OUT_DIR.resolve()}')
print()
print('  01_matriz_canales.csv          — tabla paciente × canal (0/1)')
print('  02_resumen_categorias.csv      — disponibilidad por categoría')
print('  03_saturacion_por_canal.csv    — saturación promedio por canal')
print('  04_saturacion_por_paciente.csv — saturación detallada por paciente')
print('  05_frecuencias_muestreo.csv    — sfreq y duración por archivo')
print('  06_inventario_canales_raw.csv  — inventario completo sin procesar')